In [0]:
ls /Volumes/workspace/sp100_stock_price/sp100data/

In [0]:
# Reads CSV data stored in a DBFS using the dataframe
df_bronze = spark.read.format("csv").option("header", True).load(
    "/Volumes/workspace/sp100_stock_price/sp100data/sp100_historical_data.csv"
)
display(df_bronze)

In [0]:
df.dtypes

In [0]:
from pyspark.sql.functions import col, lag, round, to_timestamp, avg, stddev, regexp_replace
from pyspark.sql import Window

window_spec = Window.partitionBy("ticker").orderBy("date")
window_spec_7 = Window.partitionBy("ticker").orderBy("date").rowsBetween(-6, 0)
window_spec_30 = Window.partitionBy("ticker").orderBy("date").rowsBetween(-29, 0)
vol_window = Window.partitionBy("ticker").orderBy("date").rowsBetween(-29, 0)

df_modified = (
    df_bronze.withColumn("Date",to_timestamp(regexp_replace(col("Date"), r'([+-]\d{2}:\d{2})$', ''),"yyyy-MM-dd HH:mm:ss")) # remove offset
      .withColumn("Open", col("Open").cast("double"))
      .withColumn("High", col("High").cast("double"))
      .withColumn("Low", col("Low").cast("double"))
      .withColumn("Close", col("Close").cast("double"))
      .withColumn("Volume", col("Volume").cast("double"))
      # Rename columns to lowercase
      .withColumnRenamed("Date", "date")
      .withColumnRenamed("Open", "open")
      .withColumnRenamed("High", "high")
      .withColumnRenamed("Low", "low")
      .withColumnRenamed("Close", "close")
      .withColumnRenamed("Volume", "volume")
      # Previous close & daily return
      .withColumn("prev_close", lag(col("close")).over(window_spec))
      .withColumn(
          "daily_return_pct",
          round(((col("close") - col("prev_close")) / col("prev_close")) * 100, 4)
      )
      # Moving averages
      .withColumn("ma7", avg(col("close")).over(window_spec_7))
      .withColumn("ma30", avg(col("close")).over(window_spec_30))
      # 30-day rolling volatility
      .withColumn("volatility_30d", stddev(col("daily_return_pct")).over(vol_window))
)

display(df_modified)


In [0]:
df_modified.dtypes

In [0]:
df_modified.count()

In [0]:
from pyspark.sql.functions import col, when, abs, lag
df_quality = df_modified.withColumn(
    "quality_flag",
    when(
        (col("date").isNull()) | 
        (col("open") <= 0) | 
        (col("close") <= 0) | 
        (col("volume") < 0),
        "BAD"
    ).when(
        (col("prev_close").isNotNull()) & 
        (abs(col("close") - col("prev_close")) / col("prev_close") > 0.2),  # >20% daily spike
        "WARNING"
    ).otherwise("OK")
)

In [0]:
df_silver = df_quality.filter(col("volume").isNotNull())\
    .filter(col("close").isNotNull())

In [0]:
df_silver.count()

In [0]:
df_silver.filter(col("date").isNull()).count()

In [0]:
display(df_silver)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("ticker") \
    .save("/Volumes/workspace/sp100_stock_price/sp100data/silver/s&p100_clean")

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/sp100_stock_price/sp100data/silver/s&p100_clean/`

In [0]:
# Lists all companies available on the dataset using pandas
#tickers = [row['ticker'] for row in df.select('ticker').distinct().collect()]
#display(tickers)

In [0]:
#Reads list of companies in full
#df.createOrReplaceTempView("sp100")
#names_df = spark.sql("select distinct name from sp100")
#display(names_df)

In [0]:
#from pyspark.sql import Window
#from pyspark.sql.functions import col, lag, round

#window_spec = Window.partitionBy("ticker").orderBy("date")

#df_casted = df.withColumn("close", col("close").cast("double"))

#df_with_prev = df_casted.withColumn(
#    "prev_close", lag(col("close")).over(window_spec)
#)

#df_with_return = df_with_prev.withColumn(
#    "daily_return_pct",
#    round(
#        ((col("close") - col("prev_close")) / col("prev_close")) * 100, 4
#    )
#)

#display(df_with_return)

In [0]:
"""from pyspark.sql.functions import year, max as spark_max, min as spark_min, col

df_with_year = df_with_return.withColumn("year", year(col("date")))

high_low_df = (
    df_with_year.groupBy("ticker", "year")
    .agg(
        spark_max("close").alias("highest_close"),
        spark_min("close").alias("lowest_close")
    )
)

display(high_low_df)"""

In [0]:
"""""df_2025 = high_low_df.filter(col("year") == 2025)
display(df_2025)"""

In [0]:
# Join to get all columns for highest close
"""highest_df = df_with_year.join(
    high_low_df,
    (df_with_year["ticker"] == high_low_df["ticker"]) &
    (df_with_year["year"] == high_low_df["year"]) &
    (df_with_year["highest_close"] == high_low_df["highest_close"])
).select(df_with_year["*"])

# Join to get all columns for lowest close
lowest_df = df_with_year.join(
    high_low_df,
    (df_with_year["ticker"] == high_low_df["ticker"]) &
    (df_with_year["year"] == high_low_df["year"]) &
    (df_with_year["lowest_close"] == high_low_df["lowest_close"])
).select(df_with_year["*"])

# Union both DataFrames and remove duplicates if a row is both highest and lowest
result_df = highest_df.unionByName(lowest_df).dropDuplicates()

display(result_df)"""